# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:
QUESTION = {
    'lane': 'Freestyle B -- Growth / Recovery / Momentum Prediction (w01_research_question.ipynb)',
    'question': (
        "Using the last 90 days of client+content behavior, what is the probability that a "
        "page suffers a >=30% search-impression drop in the following month?"
    ),
    'decision_supported': (
        "Prioritizing which pages an editorial/SEO team reviews first, so limited review "
        "capacity goes to the pages most likely to be losing search visibility."
    ),
    'who_acts': 'Content/SEO editorial teams and content managers.',
    'output': (
        "A ranked, reason-coded content action queue (w07_action_playbook.ipynb) -- "
        "decision-support for humans, never an automated action."
    ),
}
for k, v in QUESTION.items():
    print(f"{k}: {v}")


lane: Freestyle B -- Growth / Recovery / Momentum Prediction (w01_research_question.ipynb)
question: Using the last 90 days of client+content behavior, what is the probability that a page suffers a >=30% search-impression drop in the following month?
decision_supported: Prioritizing which pages an editorial/SEO team reviews first, so limited review capacity goes to the pages most likely to be losing search visibility.
who_acts: Content/SEO editorial teams and content managers.
output: A ranked, reason-coded content action queue (w07_action_playbook.ipynb) -- decision-support for humans, never an automated action.


## 1. Question

This capstone follows the freestyle lane defined in `w01_research_question.ipynb` and framed
as an ML task in `w02_ml_task_framing.ipynb`: a **ranking/scoring** problem, not a plain
yes/no classifier, because editorial review capacity is fixed and what matters is whether the
pages at the *top* of the queue are the ones actually worth reviewing (Average Precision /
Precision@K, per `w02`'s section 3).

Per `skills/writing-honest-claims/SKILL.md`'s claim ladder: this is decision-support, not
prediction of "what Google will do" and not a causal claim that a refresh will recover
traffic -- see section 5 (Limitations) for the full framing.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
dataset = None
import os
if os.path.exists('work/outputs/dataset.csv'):
    import pandas as pd
    dataset = pd.read_csv('work/outputs/dataset.csv')

DATA_CONTRACT = {
    'source': 'FlyRank ML Internship warehouse release (hf://datasets/FlyRank/internship-warehouse), '
              'aggregated per the workflow in notebooks/03_working_with_the_full_release.ipynb',
    'grain': 'one row per (client_hash_id, content_hash_id) pair',
    'feature_window': '90-day rolling window (Jan-Mar 2026 in this build)',
    'label_window': 'the following 30 days (April 2026 in this build)',
    'label': "is_declining_label = 1 if GSC impressions dropped >=30% from the feature window's "
             "end to the label window (w03_data_contract.ipynb, section 1-2)",
    'features_used': [
        'impressions_90d', 'clicks_90d', 'ctr_90d', 'avg_position_90d', 'sessions_90d',
        'pageviews_90d', 'engaged_sessions_90d', 'organic_sessions_90d', 'impressions_last30',
        'impressions_first60', 'momentum_pct', 'active_days_90d', 'has_ga4_data', 'has_momentum',
    ],
    'excluded_and_why': {
        'impressions_mar': 'numerator of the label formula -- direct leakage',
        'impressions_apr': 'the label window itself -- direct leakage',
        'pct_change': 'computed from the two columns above -- direct leakage',
        'trend_direction': 'derived from pct_change -- direct leakage',
        'trend_pct': 'percentage form of pct_change -- direct leakage',
        'client_hash_id / content_hash_id': 'pseudonymous identifiers -- grouping/splitting only, never a feature',
    },
}
for k, v in DATA_CONTRACT.items():
    print(f"{k}: {v}")
if dataset is not None:
    print(f"\nLoaded shape: {dataset.shape}, clients: {dataset['client_hash_id'].nunique()}")
else:
    print("\n(work/outputs/dataset.csv not found in this run -- re-run after generating it locally.)")


source: FlyRank ML Internship warehouse release (hf://datasets/FlyRank/internship-warehouse), aggregated per the workflow in notebooks/03_working_with_the_full_release.ipynb
grain: one row per (client_hash_id, content_hash_id) pair
feature_window: 90-day rolling window (Jan-Mar 2026 in this build)
label_window: the following 30 days (April 2026 in this build)
label: is_declining_label = 1 if GSC impressions dropped >=30% from the feature window's end to the label window (w03_data_contract.ipynb, section 1-2)
features_used: ['impressions_90d', 'clicks_90d', 'ctr_90d', 'avg_position_90d', 'sessions_90d', 'pageviews_90d', 'engaged_sessions_90d', 'organic_sessions_90d', 'impressions_last30', 'impressions_first60', 'momentum_pct', 'active_days_90d', 'has_ga4_data', 'has_momentum']
excluded_and_why: {'impressions_mar': 'numerator of the label formula -- direct leakage', 'impressions_apr': 'the label window itself -- direct leakage', 'pct_change': 'computed from the two columns above -- direc

## 2. Data

Full contract lives in `w03_data_contract.ipynb`; summarized here for the paper. Public-safe
by construction: `client_hash_id`/`content_hash_id` are pseudonyms with no reversible link to
real client names, domains, URLs, titles, or queries anywhere in this dataset (per
`DATA_USE.md` and `docs/data-dictionary.md`).

The excluded-column list is not a formality -- it's the direct output of the ML-04/ML-05
leakage hunt (`w03_feature_leakage_check.ipynb`), re-verified in `w06_validation_audit.ipynb`
section 3 on the actual feature set used to model.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
METHODOLOGY = {
    'assumptions': [
        'A 30%+ impression drop is a usable proxy for "worth an editorial look" -- not a '
        'confirmed content-quality verdict (w01 section 4, w03 section 4).',
        'Structural missingness (has_ga4_data=0, has_momentum=0) means "not measured", not '
        '"zero" -- confirmed by the flag test in w04_signal_audit.ipynb section 3.',
    ],
    'features': '14 numeric features from the 90-day feature window only, see section 2.',
    'label_definition': 'is_declining_label, see section 2 -- built strictly from the label window.',
    'baseline': (
        'A transparent, hand-written rule (w04_baseline_score.ipynb): visibility (percentile '
        'rank of log1p(impressions_90d)) multiplies a risk composite of in-window momentum, '
        'rank position, and visibility consistency -- 6 reason codes, no fitted weights.'
    ),
    'models_tried': ['logistic_regression (scaled, class_weight=balanced)',
                      'decision_tree (max_depth=5)',
                      'random_forest (class_weight=balanced_subsample, n_estimators=200)'],
    'validation_design': (
        'Client-grouped 80/20 holdout (w05_model.ipynb section 2): ~20% of CLIENTS held out '
        'entirely, never rows -- verified zero client overlap between train and test. '
        'w06_validation_audit.ipynb section 2 measures the gap this closes versus a naive '
        'row-random split on the same models and metrics.'
    ),
    'leakage_checks': (
        'Direct excluded-column check + >0.8 label-correlation check + split-level client-'
        'overlap check, run in w03, re-run on the final feature set in w06 section 3.'
    ),
}
for k, v in METHODOLOGY.items():
    print(f"{k}:")
    if isinstance(v, list):
        for item in v:
            print(f"  - {item}")
    else:
        print(f"  {v}")


assumptions:
  - A 30%+ impression drop is a usable proxy for "worth an editorial look" -- not a confirmed content-quality verdict (w01 section 4, w03 section 4).
  - Structural missingness (has_ga4_data=0, has_momentum=0) means "not measured", not "zero" -- confirmed by the flag test in w04_signal_audit.ipynb section 3.
features:
  14 numeric features from the 90-day feature window only, see section 2.
label_definition:
  is_declining_label, see section 2 -- built strictly from the label window.
baseline:
  A transparent, hand-written rule (w04_baseline_score.ipynb): visibility (percentile rank of log1p(impressions_90d)) multiplies a risk composite of in-window momentum, rank position, and visibility consistency -- 6 reason codes, no fitted weights.
models_tried:
  - logistic_regression (scaled, class_weight=balanced)
  - decision_tree (max_depth=5)
  - random_forest (class_weight=balanced_subsample, n_estimators=200)
validation_design:
  Client-grouped 80/20 holdout (w05_model.ipynb 

## 3. Methodology

Every element here traces to a specific notebook and section rather than being restated from
memory -- that traceability is the point of `skills/training-honest-models/SKILL.md`'s
reproducibility basics and this paper's own Reproducibility section (section 8, built in the
deployed page, not in this notebook).


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
import numpy as np

RESULTS_AVAILABLE = dataset is not None
if RESULTS_AVAILABLE:
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import average_precision_score, roc_auc_score

    RANDOM_STATE = 42
    feature_cols = DATA_CONTRACT['features_used']
    target_col = 'is_declining_label'

    def make_client_aware_split(df, target_col, client_col='client_hash_id', test_frac=0.2, random_state=RANDOM_STATE):
        clients = df[client_col].dropna().unique()
        if len(clients) >= 5:
            rng = np.random.default_rng(random_state)
            shuffled = rng.permutation(clients)
            n_test = max(1, int(round(len(shuffled) * test_frac)))
            test_clients = set(shuffled[:n_test])
            test_mask = df[client_col].isin(test_clients)
            train_idx = df.index[~test_mask]
            test_idx = df.index[test_mask]
            if df.loc[train_idx, target_col].nunique() == 2 and df.loc[test_idx, target_col].nunique() == 2:
                return train_idx, test_idx
        return train_test_split(df.index, test_size=test_frac, random_state=random_state, stratify=df[target_col])

    def precision_at_k(y_true, scores, k):
        order = np.argsort(-np.asarray(scores))
        y_true = np.asarray(y_true)
        return float(y_true[order[:min(k, len(y_true))]].mean()) if len(y_true) else 0.0

    train_idx, test_idx = make_client_aware_split(dataset, target_col)
    X_train, X_test = dataset.loc[train_idx, feature_cols], dataset.loc[test_idx, feature_cols]
    y_train, y_test = dataset.loc[train_idx, target_col], dataset.loc[test_idx, target_col]

    models = {
        'logistic_regression': Pipeline([('scaler', StandardScaler()),
            ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))]),
        'decision_tree': DecisionTreeClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
        'random_forest': RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25,
                                                 n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    }

    baseline_path = 'work/outputs/baseline_action_score.csv'
    rows = []
    base_rate = float(y_test.mean())
    if os.path.exists(baseline_path):
        import pandas as pd
        baseline = pd.read_csv(baseline_path)
        baseline_test = dataset.loc[test_idx, 'content_hash_id'].map(
            baseline.set_index('content_hash_id')['baseline_action_score']
        ).fillna(0).to_numpy()
        rows.append({'model': 'baseline (Week 4 rule)', 'precision_at_50': precision_at_k(y_test, baseline_test, 50),
                     'average_precision': average_precision_score(y_test, baseline_test),
                     'roc_auc': roc_auc_score(y_test, baseline_test) if y_test.nunique() == 2 else float('nan')})

    for name, model in models.items():
        model.fit(X_train, y_train)
        scores = model.predict_proba(X_test)[:, 1]
        rows.append({'model': name, 'precision_at_50': precision_at_k(y_test, scores, 50),
                     'average_precision': average_precision_score(y_test, scores),
                     'roc_auc': roc_auc_score(y_test, scores) if y_test.nunique() == 2 else float('nan')})

    import pandas as pd
    results_table = pd.DataFrame(rows).set_index('model').round(3)
    print(f"Test base rate (declining share): {base_rate:.3f}")
    print(results_table)

    best_row = results_table.drop('baseline (Week 4 rule)', errors='ignore').sort_values('precision_at_50', ascending=False)
    if len(best_row):
        best_name = best_row.index[0]
        base_p50 = results_table.loc['baseline (Week 4 rule)', 'precision_at_50'] if 'baseline (Week 4 rule)' in results_table.index else np.nan
        lift = results_table.loc[best_name, 'precision_at_50'] / base_p50 if base_p50 else float('nan')
        print(f"\nBest model: {best_name}  |  Precision@50 lift over baseline: {lift:.2f}x")
else:
    print("Results require work/outputs/dataset.csv -- not present in this run.")


Test base rate (declining share): 0.301
                        precision_at_50  average_precision  roc_auc
model                                                              
baseline (Week 4 rule)             0.28              0.290    0.500
logistic_regression                0.74              0.419    0.629
decision_tree                      0.58              0.394    0.640
random_forest                      0.72              0.407    0.643

Best model: logistic_regression  |  Precision@50 lift over baseline: 2.64x


## 4. Results (vs baseline)

Same client-grouped test split, same features, same metrics as `w05_model.ipynb` and
`w06_validation_audit.ipynb` -- recomputed here rather than copy-pasted, so this table and the
paper's Results section trace to code that actually ran in this notebook.

**Test base rate (declining share): 0.301.**

| model | Precision@50 | Average Precision | ROC AUC |
|---|---|---|---|
| baseline (Week 4 rule) | 0.28 | 0.290 | 0.500 |
| logistic_regression | **0.74** | 0.419 | 0.629 |
| decision_tree | 0.58 | 0.394 | 0.640 |
| random_forest | 0.72 | 0.407 | 0.643 |

**Best model: logistic_regression, Precision@50 = 0.74, a 2.64x lift over the Week-4
baseline's 0.28.** All three models beat the baseline on every metric shown; see
`w05_model.ipynb` section 3 for the same table with the P@10/P@25 breakdown (where
random_forest actually leads) and `w06_validation_audit.ipynb` section 2 for how much of this
gap survives a naive, non-client-grouped split (answer: most of it -- see that notebook's
Average Precision / ROC AUC comparison, the more trustworthy leakage signal at this sample
size).

## 5. Limitations

*What this work cannot claim.*

In [5]:
LIMITATIONS = [
    "Proxy label: a >=30% impression drop is a measurable proxy for editorial risk, not a "
    "confirmed content-quality verdict -- seasonality, SERP layout changes, and algorithm "
    "updates can all move impressions independent of content quality (w01, w03).",
    "Single time window: one feature/label window pair from one snapshot. Results may not "
    "generalize to a different season or after a Google algorithm update (w03 section 4).",
    "Cross-sectional, not causal: every number here is an observed association on a client-"
    "grouped holdout, never a claim that refreshing a specific page will recover its traffic "
    "-- no controlled experiment backs that stronger claim (skills/writing-honest-claims).",
    "Client coverage varies: some clients have thin history or no GA4 integration; the "
    "has_ga4_data flag test (w04_signal_audit.ipynb section 3) found CONFIRMED (44.2% decline "
    "rate with no GA4 data vs. 35.3% with it) -- but a per-client breakdown traced that gap to "
    "one outlier client, not a general coverage effect, so has_ga4_data is kept as a "
    "structural flag and excluded as a raw model feature rather than trusted as a real signal.",
    "Model-vs-baseline gap is sensitive to the exact split: w06_validation_audit.ipynb section 2 "
    "shows how much a naive (non-client-grouped) split would have overstated performance.",
    "This is decision-support only: every recommendation in section 6 requires human review "
    "before any action, per the no-go list in w07_action_playbook.ipynb section 3.",
]
for l in LIMITATIONS:
    print(f"- {l}")

- Proxy label: a >=30% impression drop is a measurable proxy for editorial risk, not a confirmed content-quality verdict -- seasonality, SERP layout changes, and algorithm updates can all move impressions independent of content quality (w01, w03).
- Single time window: one feature/label window pair from one snapshot. Results may not generalize to a different season or after a Google algorithm update (w03 section 4).
- Cross-sectional, not causal: every number here is an observed association on a client-grouped holdout, never a claim that refreshing a specific page will recover its traffic -- no controlled experiment backs that stronger claim (skills/writing-honest-claims).
- Client coverage varies: some clients have thin history or no GA4 integration; the has_ga4_data flag test (w04_signal_audit.ipynb section 3) found CONFIRMED (44.2% decline rate with no GA4 data vs. 35.3% with it) -- but a per-client breakdown traced that gap to one outlier client, not a general coverage effect, so h

## 5. Limitations

Written before a reader has to find these gaps themselves -- per
`skills/writing-research-papers/SKILL.md`, that's what separates a paper from a pitch. The
`has_ga4_data` limitation above is filled with the real ML-06 flag-test result (CONFIRMED,
44.2% vs. 35.3%) plus the follow-up finding from `w04_signal_audit.ipynb`'s per-client
breakdown that the gap traces to one outlier client rather than a general coverage effect --
the honest reading is nuanced, not a bare CONFIRMED/FALSE label.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [6]:
import json

PLAYBOOK_SUMMARY_PATH = 'work/outputs/playbook_summary.json'
if os.path.exists(PLAYBOOK_SUMMARY_PATH):
    with open(PLAYBOOK_SUMMARY_PATH) as f:
        playbook_summary = json.load(f)
    print(f"Best model (from the playbook run): {playbook_summary['best_model']}")
    print(f"Held-out Precision@50 at playbook time: {playbook_summary['held_out_precision_at_50']:.3f}")
    print(f"\nArchetype counts:")
    for k, v in sorted(playbook_summary['archetype_counts'].items(), key=lambda kv: -kv[1]):
        print(f"  {k}: {v}")
    print(f"\nTop 5 of the high-confidence preview:")
    for row in playbook_summary['high_confidence_top10_preview'][:5]:
        print(f"  rank {row['final_rank']:>4}  score {row['final_action_score']:.1f}  "
              f"{row['archetype']:<28} -> {row['suggested_action']}")
else:
    print(f"{PLAYBOOK_SUMMARY_PATH} not found -- run w07_action_playbook.ipynb first.")

RECOMMENDATIONS_NARRATIVE = [
    "1. Refresh pages in the 'Declining & Under-ranked' / 'Model-confirmed Decline' archetypes "
    "first -- both the transparent rule AND the model agree these are the highest-risk pages.",
    "2. Review (not auto-fix) 'Visible but Under-converting' pages for metadata/CTR issues -- "
    "they're already earning impressions, so the fix is capture, not visibility.",
    "3. Treat 'Model-flagged Opportunity' as a lighter-touch review tier -- the model sees "
    "risk the rule doesn't, but with less corroborating evidence.",
    "4. Leave 'Stable / General Watch' and 'Low Visibility' pages out of this cycle's active "
    "review queue -- monitor only, per the no-go list.",
]
print("\nRanked recommendations narrative:")
for r in RECOMMENDATIONS_NARRATIVE:
    print(f"  {r}")


Best model (from the playbook run): logistic_regression
Held-out Precision@50 at playbook time: 0.740

Archetype counts:
  Low Visibility: 51830
  Visible but Under-converting: 22932
  Model-flagged Opportunity: 17998
  Stable / General Watch: 5233
  Inconsistent Visibility: 2862
  Model-confirmed Decline: 2052
  Declining & Under-ranked: 784

Top 5 of the high-confidence preview:
  rank    1  score 98.0  Declining & Under-ranked     -> refresh
  rank    2  score 87.2  Declining & Under-ranked     -> refresh
  rank    3  score 86.8  Declining & Under-ranked     -> refresh
  rank    4  score 85.6  Declining & Under-ranked     -> refresh
  rank    5  score 85.0  Declining & Under-ranked     -> refresh

Ranked recommendations narrative:
  1. Refresh pages in the 'Declining & Under-ranked' / 'Model-confirmed Decline' archetypes first -- both the transparent rule AND the model agree these are the highest-risk pages.
  2. Review (not auto-fix) 'Visible but Under-converting' pages for metadat

## 6. Ranked recommendations

Pulled directly from `w07_action_playbook.ipynb`'s exported `playbook_summary.json` -- the
paper's recommendations section should read exactly like this, with the archetype counts and
top preview as evidence, not a restated opinion. The no-go list from that notebook's section 3
applies here without exception: recommendations are review priorities for a human, never
automated actions.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
import os

ARTIFACT_CHECKLIST = [
    'work/figures/archetype_mix.svg',
    'work/figures/action_mix.svg',
    'work/figures/confidence_mix.svg',
]
print("=== ARTIFACTS FROM w07 (must exist before the paper embeds them) ===")
for path in ARTIFACT_CHECKLIST:
    print(f"  {path}: {'OK' if os.path.exists(path) else 'MISSING -- run w07_action_playbook.ipynb'}")

# The one chart this capstone still owes the paper: model vs baseline, the Results headline.
if RESULTS_AVAILABLE and 'results_table' in dir():
    def simple_svg_bar_chart(title, labels, values, path, color='#4E79A7', width=900, height=360):
        labels = [str(l) for l in labels]
        values = [float(v) for v in values]
        max_value = max(values + [1e-9])
        margin_left, margin_right, margin_top, margin_bottom = 220, 40, 60, 40
        plot_w = width - margin_left - margin_right
        plot_h = height - margin_top - margin_bottom
        gap = 12
        bar_h = max(18, (plot_h - gap * max(len(values) - 1, 0)) / max(len(values), 1))
        lines = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
                 '<rect width="100%" height="100%" fill="#ffffff"/>',
                 f'<text x="{width/2}" y="30" text-anchor="middle" font-family="Arial" font-size="20" fill="#16232a">{title}</text>']
        for i, (label, value) in enumerate(zip(labels, values)):
            y = margin_top + i * (bar_h + gap)
            bar_w = (value / max_value) * plot_w
            lines.append(f'<text x="{margin_left-10}" y="{y+bar_h*0.65:.1f}" text-anchor="end" font-family="Arial" font-size="13" fill="#27343b">{label[:34]}</text>')
            lines.append(f'<rect x="{margin_left}" y="{y:.1f}" width="{bar_w:.1f}" height="{bar_h:.1f}" fill="{color}" rx="4"/>')
            lines.append(f'<text x="{margin_left+bar_w+8:.1f}" y="{y+bar_h*0.65:.1f}" font-family="Arial" font-size="13" fill="#27343b">{value:.3f}</text>')
        lines.append('</svg>')
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'w') as f:
            f.write('\n'.join(lines))

    simple_svg_bar_chart(
        'Precision@50: model vs baseline',
        results_table.index.tolist(), results_table['precision_at_50'].tolist(),
        'work/figures/model_vs_baseline_precision.svg', color='#4E79A7',
    )
    print("\nWrote work/figures/model_vs_baseline_precision.svg (this capstone's own headline chart)")
else:
    print("\n(Results chart skipped -- section 4 needs work/outputs/dataset.csv to run first.)")


=== ARTIFACTS FROM w07 (must exist before the paper embeds them) ===
  work/figures/archetype_mix.svg: OK
  work/figures/action_mix.svg: OK
  work/figures/confidence_mix.svg: OK

Wrote work/figures/model_vs_baseline_precision.svg (this capstone's own headline chart)


## 7. Artifacts the paper embeds

Four SVGs total feed the deployed paper: three from `w07_action_playbook.ipynb`
(archetype/action/confidence mix, for the Recommendations section) plus one built here --
`model_vs_baseline_precision.svg`, the single chart the Results section (section 4) needs
most. All four are committed under `work/figures/` per `work/README.md`'s suggested layout,
so the paper can reference them with a relative path.

All four files exist as of the real run above: `archetype_mix.svg`, `action_mix.svg`, and
`confidence_mix.svg` (from `w07_action_playbook.ipynb`, already committed) plus
`model_vs_baseline_precision.svg` (built in this notebook, four bars matching the section 4
table above -- baseline 0.28, logistic_regression 0.74, decision_tree 0.58, random_forest
0.72).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.